In [6]:
# vector_services/vector_visualizer.py

import numpy as np
from plotly.offline import plot
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1) Try scikit-learn TSNE
use_tsne = False
try:
    from sklearn.manifold import TSNE
    use_tsne = True
    print("Using scikit‑learn TSNE for dimensionality reduction.")
except ImportError:
    # 2) Fallback to openTSNE
    try:
        from openTSNE import TSNE
        use_tsne = True
        print("Using openTSNE for dimensionality reduction.")
    except ImportError:
        TSNE = None
        print("TSNE unavailable; please install scikit‑learn or openTSNE.")

# 3) If no TSNE, try UMAP
use_umap = False
if not use_tsne:
    try:
        import umap
        use_umap = True
        print("Using UMAP for dimensionality reduction.")
    except ImportError:
        print("UMAP unavailable.")

# 4) If still no reducer, fall back to PCA
if not use_tsne and not use_umap:
    print("Falling back to PCA (NumPy SVD) for dimensionality reduction.")

# 5) Import your colour mapping
from data_curator import assign_color, safe_embeddings_constructor


class VectorVisualizer:
    def __init__(self, vector_store):
        """
        Initialize with a Chroma collection or a dict containing "collection".
        """
        self.vector_store = vector_store
        self.collection = self._get_collection()

    def _get_collection(self):
        if isinstance(self.vector_store, dict) and "collection" in self.vector_store:
            return self.vector_store["collection"]
        elif hasattr(self.vector_store, "_collection"):
            return self.vector_store._collection
        return self.vector_store

    def _reduce(self, vectors: np.ndarray, n_components: int) -> np.ndarray:
        n, dim = vectors.shape
        if n < max(n_components + 1, 5):
            raise ValueError(f"Need at least {max(n_components+1,5)} samples; got {n}.")

        if use_tsne and TSNE is not None:
            return TSNE(
                n_components=n_components,
                perplexity=min(30, max(1, n - 1)),
                random_state=42
            ).fit_transform(vectors)

        if use_umap:
            return umap.UMAP(
                n_components=n_components,
                n_neighbors=min(15, max(2, n - 1)),
                random_state=42
            ).fit_transform(vectors)

        # PCA fallback
        X = vectors - vectors.mean(axis=0)
        U, S, Vt = np.linalg.svd(X, full_matrices=False)
        W = Vt[:n_components].T
        return X.dot(W)

    def visualize_2d(self):
        result = self.collection.get(include=['embeddings','documents','metadatas'])
        vecs = np.array(result['embeddings'])
        docs = result['documents']
        metas = result.get('metadatas', [{}]*len(docs))

        if vecs.shape[0] < 5:
            print("Not enough samples to visualize (need ≥5).")
            return

        colors = [ assign_color(md.get("doc_type","other")) for md in metas ]
        hovers = [
            f"Type: {md.get('doc_type','other')}<br>{doc[:100]}…"
            for md, doc in zip(metas, docs)
        ]

        try:
            red2 = self._reduce(vecs, n_components=2)
            fig = go.Figure(go.Scatter(
                x=red2[:,0], y=red2[:,1],
                mode='markers',
                marker=dict(size=6, color=colors, opacity=0.8),
                text=hovers, hoverinfo='text'
            ))
            method = "TSNE" if use_tsne else ("UMAP" if use_umap else "PCA")
            fig.update_layout(title=f"2D Visualization ({method})", width=900, height=700)
            plot(fig, auto_open=True)
        except Exception as e:
            print(f"Error in 2D visualization: {e}")

    def visualize_3d(self):
        result = self.collection.get(include=['embeddings','documents','metadatas'])
        vecs = np.array(result['embeddings'])
        docs = result['documents']
        metas = result.get('metadatas', [{}]*len(docs))

        if vecs.shape[0] < 5:
            print("Not enough samples to visualize (need ≥5).")
            return

        colors = [ assign_color(md.get("doc_type","other")) for md in metas ]
        hovers = [
            f"Type: {md.get('doc_type','other')}<br>{doc[:100]}…"
            for md, doc in zip(metas, docs)
        ]

        try:
            red3 = self._reduce(vecs, n_components=3)
            fig = go.Figure(go.Scatter3d(
                x=red3[:,0], y=red3[:,1], z=red3[:,2],
                mode='markers',
                marker=dict(size=5, color=colors, opacity=0.8),
                text=hovers, hoverinfo='text'
            ))
            method = "TSNE" if use_tsne else ("UMAP" if use_umap else "PCA")
            fig.update_layout(title=f"3D Visualization ({method})", width=900, height=700)
            plot(fig, auto_open=True)
        except Exception as e:
            print(f"Error in 3D visualization: {e}")

    def visualize_both(self):
        result = self.collection.get(include=['embeddings','documents','metadatas'])
        vecs = np.array(result['embeddings'])
        docs = result['documents']
        metas = result.get('metadatas', [{}]*len(docs))

        if vecs.shape[0] < 5:
            print("Not enough samples to visualize (need ≥5).")
            return

        colors = [ assign_color(md.get("doc_type","other")) for md in metas ]
        hovers = [
            f"Type: {md.get('doc_type','other')}<br>{doc[:100]}…"
            for md, doc in zip(metas, docs)
        ]

        try:
            red2 = self._reduce(vecs, n_components=2)
            red3 = self._reduce(vecs, n_components=3)

            trace2d = go.Scatter(
                x=red2[:,0], y=red2[:,1],
                mode='markers',
                marker=dict(size=5, color=colors, opacity=0.8),
                text=hovers, hoverinfo='text'
            )
            trace3d = go.Scatter3d(
                x=red3[:,0], y=red3[:,1], z=red3[:,2],
                mode='markers',
                marker=dict(size=5, color=colors, opacity=0.8),
                text=hovers, hoverinfo='text'
            )

            fig = make_subplots(
                rows=1, cols=2,
                subplot_titles=("2D View", "3D View"),
                specs=[[{"type":"scatter"}, {"type":"scatter3d"}]]
            )
            fig.add_trace(trace2d, row=1, col=1)
            fig.add_trace(trace3d, row=1, col=2)

            method = "TSNE" if use_tsne else ("UMAP" if use_umap else "PCA")
            fig.update_layout(title=f"Combined Visualizations ({method})", width=1400, height=650)
            plot(fig, auto_open=True)
        except Exception as e:
            print(f"Error in combined visualization: {e}")


if __name__ == "__main__":
    from langchain_chroma import Chroma
    store = Chroma(
        persist_directory="./usiu_vector_db",
        embedding_function=safe_embeddings_constructor()
    )
    viz = VectorVisualizer(store)
    print("Running 2D…"); viz.visualize_2d()
    print("Running 3D…"); viz.visualize_3d()
    print("Running both…"); viz.visualize_both()


TSNE unavailable; please install scikit‑learn or openTSNE.
UMAP unavailable.
Falling back to PCA (NumPy SVD) for dimensionality reduction.
Running 2D…
Running 3D…
Running both…


In [1]:
# vector_services/vector_visualizer.py

# ── Monkey‑patch to satisfy "from numpy.core.numeric import ComplexWarning" in sklearn ──
import numpy as _np
import numpy.core.numeric as _numeric
if not hasattr(_numeric, "ComplexWarning"):
    class ComplexWarning(Warning):
        """Placeholder for numpy.core.numeric.ComplexWarning"""
        pass
    _numeric.ComplexWarning = ComplexWarning

# ────────────────────────────────────────────────────────────────────────────────

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.manifold import TSNE

from data_curator import assign_color, safe_embeddings_constructor


class VectorVisualizer:
    def __init__(self, vector_store):
        """
        Initialize the visualizer with an existing vector store.
        The vector store can either be a dict with a "collection" key
        or a Chroma instance (which has a _collection attribute).
        """
        self.vector_store = vector_store
        self.collection = self._get_collection()

    def _get_collection(self):
        # If vector_store is a dict with a "collection" key, return that.
        if isinstance(self.vector_store, dict) and "collection" in self.vector_store:
            return self.vector_store["collection"]
        # If it has an attribute _collection, use that.
        elif hasattr(self.vector_store, "_collection"):
            return self.vector_store._collection
        # Otherwise, assume the vector_store is already the collection.
        return self.vector_store

    def visualize_2d(self):
        result = self.collection.get(include=['embeddings', 'documents', 'metadatas'])
        vectors = np.array(result['embeddings'])
        documents = result['documents']
        metadatas = result.get('metadatas', [{}] * len(documents))
        n_samples = vectors.shape[0]
        if n_samples < 5:
            print("Not enough samples to visualize TSNE.")
            return

        perplexity = min(30, max(1, n_samples - 1))
        # Use "doc_type" for color mapping and text.
        colours = [assign_color((md or {}).get("doc_type", "other")) for md in metadatas]
        doc_types = [(md or {}).get("doc_type", "other") for md in metadatas]
        text = [f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)]
        
        tsne = TSNE(n_components=2, random_state=42, perplexity=perplexity)
        reduced_vectors = tsne.fit_transform(vectors)
        fig = go.Figure(data=[go.Scatter(
            x=reduced_vectors[:, 0],
            y=reduced_vectors[:, 1],
            mode='markers',
            marker=dict(size=5, color=colours, opacity=0.8),
            text=text,
            hoverinfo='text'
        )])
        fig.update_layout(title="2D Vector Store Visualization", width=1000, height=800)
        fig.show()

    def visualize_3d(self):
        result = self.collection.get(include=['embeddings', 'documents', 'metadatas'])
        vectors = np.array(result['embeddings'])
        documents = result['documents']
        metadatas = result.get('metadatas', [{}] * len(documents))
        n_samples = vectors.shape[0]
        if n_samples < 5:
            print("Not enough samples to visualize TSNE.")
            return

        perplexity = min(30, max(1, n_samples - 1))
        colours = [assign_color((md or {}).get("doc_type", "other")) for md in metadatas]
        doc_types = [(md or {}).get("doc_type", "other") for md in metadatas]
        text = [f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)]
        
        tsne = TSNE(n_components=3, random_state=42, perplexity=perplexity)
        reduced_vectors = tsne.fit_transform(vectors)
        fig = go.Figure(data=[go.Scatter3d(
            x=reduced_vectors[:, 0],
            y=reduced_vectors[:, 1],
            z=reduced_vectors[:, 2],
            mode='markers',
            marker=dict(size=5, color=colours, opacity=0.8),
            text=text,
            hoverinfo='text'
        )])
        fig.update_layout(title="3D Vector Store Visualization", width=1000, height=800)
        fig.show()

    def visualize_both(self):
        result = self.collection.get(include=['embeddings', 'documents', 'metadatas'])
        vectors = np.array(result['embeddings'])
        documents = result['documents']
        metadatas = result.get('metadatas', [{}] * len(documents))
        n_samples = vectors.shape[0]
        if n_samples < 5:
            print("Not enough samples to visualize TSNE.")
            return

        perplexity = min(30, max(1, n_samples - 1))
        colours = [assign_color((md or {}).get("doc_type", "other")) for md in metadatas]
        doc_types = [(md or {}).get("doc_type", "other") for md in metadatas]
        text = [f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)]
        
        tsne_2d = TSNE(n_components=2, random_state=42, perplexity=perplexity)
        reduced_2d = tsne_2d.fit_transform(vectors)
        trace2d = go.Scatter(
            x=reduced_2d[:, 0],
            y=reduced_2d[:, 1],
            mode='markers',
            marker=dict(size=5, color=colours, opacity=0.8),
            text=text,
            hoverinfo='text'
        )
        tsne_3d = TSNE(n_components=3, random_state=42, perplexity=perplexity)
        reduced_3d = tsne_3d.fit_transform(vectors)
        trace3d = go.Scatter3d(
            x=reduced_3d[:, 0],
            y=reduced_3d[:, 1],
            z=reduced_3d[:, 2],
            mode='markers',
            marker=dict(size=5, color=colours, opacity=0.8),
            text=text,
            hoverinfo='text'
        )
        fig = make_subplots(rows=1, cols=2,
                            subplot_titles=("2D Visualization", "3D Visualization"),
                            specs=[[{"type": "scatter"}, {"type": "scatter3d"}]])
        fig.add_trace(trace2d, row=1, col=1)
        fig.add_trace(trace3d, row=1, col=2)
        fig.update_layout(title="Combined Vector Store Visualizations", width=1500, height=700)
        fig.show()


if __name__ == "__main__":
    # Load the existing vector store from the persist directory.
    from langchain_chroma import Chroma
    vector_store = Chroma(
        persist_directory="./usiu_vector_db",
        embedding_function=safe_embeddings_constructor()
    )
    
    # Initialize the visualizer with the loaded vector store.
    visualizer = VectorVisualizer(vector_store)
    
    print("Visualizing in 2D:")
    visualizer.visualize_2d()
    
    print("Visualizing in 3D:")
    visualizer.visualize_3d()
    
    print("Visualizing both 2D and 3D:")
    visualizer.visualize_both()


Visualizing in 2D:


Visualizing in 3D:


Visualizing both 2D and 3D:


In [1]:
# test_chromadb.py
print("Testing ChromaDB with OpenTelemetry...")

try:
    # Test OpenTelemetry imports
    from opentelemetry import trace
    print("✅ Successfully imported OpenTelemetry trace")
    
    # Test ChromaDB imports
    import chromadb
    print(f"✅ Successfully imported ChromaDB {chromadb.__version__}")
    
    # Test LangChain ChromaDB integration
    from langchain_chroma import Chroma
    from langchain_community.embeddings import OpenAIEmbeddings
    print("✅ Successfully imported Chroma from langchain_chroma")
    
    # Create a simple client
    client = chromadb.Client()
    print("✅ Successfully created ChromaDB client")
    
    # Create a simple collection
    collection = client.create_collection(name="test_collection")
    print("✅ Successfully created ChromaDB collection")
    
    # Add a document
    collection.add(
        documents=["This is a test document"],
        metadatas=[{"source": "test"}],
        ids=["id1"]
    )
    print("✅ Successfully added document to collection")
    
    # Query the collection
    results = collection.query(
        query_texts=["test document"],
        n_results=1
    )
    print(f"✅ Successfully queried collection: {results}")
    
    # Test LangChain integration
    embeddings = OpenAIEmbeddings()
    langchain_chroma = Chroma(
        collection_name="langchain_test",
        embedding_function=embeddings
    )
    print("✅ Successfully created LangChain Chroma instance")
    
    print("\nAll tests passed! ChromaDB is working correctly with OpenTelemetry.")
    
except Exception as e:
    print(f"❌ Error: {e}")
    import traceback
    traceback.print_exc()
    
    # Provide more debug info
    print("\nAdditional debugging information:")
    import sys
    print(f"Python version: {sys.version}")
    
    try:
        import pkg_resources
        print("\nInstalled package versions:")
        for pkg in ['opentelemetry-api', 'opentelemetry-sdk', 
                   'opentelemetry-semantic-conventions', 'chromadb', 
                   'langchain', 'langchain-chroma']:
            try:
                version = pkg_resources.get_distribution(pkg).version
                print(f"{pkg}: {version}")
            except pkg_resources.DistributionNotFound:
                print(f"{pkg}: Not installed")
    except Exception as e2:
        print(f"Error checking packages: {e2}")

Testing ChromaDB with OpenTelemetry...
✅ Successfully imported OpenTelemetry trace
✅ Successfully imported ChromaDB 0.6.3
✅ Successfully imported Chroma from langchain_chroma
✅ Successfully created ChromaDB client
✅ Successfully created ChromaDB collection


/Users/apple/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [23:42<00:00, 58.5kiB/s]


✅ Successfully added document to collection
✅ Successfully queried collection: {'ids': [['id1']], 'embeddings': None, 'documents': [['This is a test document']], 'uris': None, 'data': None, 'metadatas': [[{'source': 'test'}]], 'distances': [[0.4501386880874634]], 'included': [<IncludeEnum.distances: 'distances'>, <IncludeEnum.documents: 'documents'>, <IncludeEnum.metadatas: 'metadatas'>]}


/var/folders/x4/dd2xz8_d4fjbb7kzdq5sp8_40000gn/T/ipykernel_75248/1891920969.py:42: LangChainDeprecationWarning: The class `OpenAIEmbeddings` was deprecated in LangChain 0.0.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import OpenAIEmbeddings``.
  embeddings = OpenAIEmbeddings()


✅ Successfully created LangChain Chroma instance

All tests passed! ChromaDB is working correctly with OpenTelemetry.


In [ ]:
!python -c "from opentelemetry import trace; print('OpenTelemetry installed successfully!')"